# Импорт библиотек

In [ ]:
%pip install scikit-learn xgboost openpyxl pandas numpy matplotlib catboost

In [3]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split, RandomizedSearchCV, learning_curve, KFold, RepeatedKFold, cross_validate, cross_val_predict, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.linear_model import RidgeCV, LassoCV, ElasticNetCV
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.metrics import r2_score, mean_absolute_error
from sklearn.feature_selection import RFECV, RFE
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from catboost import CatBoostRegressor
import xgboost as xgb

file_path = r"magnetocaloric_perovskites_full_descriptors.xlsx"
out_dir = r"DCD_paper"

# Оформление графиков

In [4]:
# Константы для оформления графиков
CM_TO_INCH = 1 / 2.54
COLUMN_WIDTH = 8.5 * CM_TO_INCH

plt.rcParams.update({
    "font.family": "serif",
    "font.serif": ["Times New Roman", "DejaVu Serif"],
    "mathtext.fontset": "stix",

    "font.size": 9,
    "axes.labelsize": 10,
    "axes.titlesize": 10,
    "xtick.labelsize": 8.5,
    "ytick.labelsize": 8.5,
    "legend.fontsize": 8.5,

    "axes.linewidth": 0.9,
    "xtick.major.width": 0.8,
    "ytick.major.width": 0.8,
    "xtick.major.size": 3.5,
    "ytick.major.size": 3.5,

    "figure.dpi": 300,
    "savefig.dpi": 600,
    "savefig.bbox": "tight",
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})

COLORS = {
    "blue": "#1f4e79",
    "red": "#b22222",
    "green": "#2e7d32",
    "gray": "#666666",
    "black": "#000000",
}

FEATURE_MAP = {
    'H_ex (T)': r'$H_{\mathrm{ex}}$ (T)',
    'min:mendeleev_number': r'$N_{\mathrm{Mend, min}}$',
    'max:melting_point': r'$T_{\mathrm{m, max}}$',
    'max:density': r'$\rho_{\max}$',
    'max:hhi_r': r'$\mathrm{HHI}_{\mathrm{res, max}}$'
}


def save_figure(fig, name):
    fig.savefig(os.path.join(out_dir, f"{name}.png"), dpi=600)


def save_figure_to(fig, name, folder):
    fig.savefig(os.path.join(folder, f"{name}.png"), dpi=600)


def format_axes(ax, grid=True):
    ax.tick_params(direction="in", top=True, right=True)

    if grid:
        ax.grid(True, linestyle="--", linewidth=0.45, alpha=0.35)

    for spine in ax.spines.values():
        spine.set_linewidth(0.9)

# Функции построения графиков

In [5]:
def plot_delta_s_histogram(delta_s):
    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.hist(
        delta_s,
        bins=22,
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        alpha=0.88
    )

    ax.axvline(delta_s.mean(), color=COLORS["red"], linestyle="--",
               linewidth=1.1, label="Mean")
    ax.axvline(delta_s.median(), color=COLORS["black"], linestyle="-",
               linewidth=1.1, label="Median")

    ax.set_xlabel(r"$\Delta S_M$ (J kg$^{-1}$ K$^{-1}$)")
    ax.set_ylabel("Frequency")
    ax.set_title(r"Distribution of $\Delta S_M$")

    ax.legend(frameon=True, edgecolor="black", framealpha=1.0)
    format_axes(ax)

    fig.tight_layout(pad=0.4)
    save_figure(fig, "Fig1_delta_S_histogram")
    plt.show()


def plot_transformation_histogram(y_original):
    y_log = np.log1p(y_original)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.hist(
        y_original,
        bins=22,
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        alpha=0.88
    )

    ax.set_xlabel(r"$\Delta S_M$ (J kg$^{-1}$ K$^{-1}$)")
    ax.set_ylabel("Frequency")
    ax.set_title(r"Original $\Delta S_M$ distribution")

    format_axes(ax)
    fig.tight_layout(pad=0.4)
    save_figure(fig, "Fig2_original_distribution")
    plt.show()

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.hist(
        y_log,
        bins=22,
        color=COLORS["green"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        alpha=0.88
    )

    ax.set_xlabel(r"$\log(1+\Delta S_M)$")
    ax.set_ylabel("Frequency")
    ax.set_title(r"Log-transformed distribution")

    format_axes(ax)
    fig.tight_layout(pad=0.4)
    save_figure(fig, "Fig3_log_distribution")
    plt.show()


def plot_rfecv_curve(rfecv, optimal_k):
    if rfecv is None:
        print("RFECV plot skipped: RFECV was not performed.")
        return

    mean_scores = rfecv.cv_results_["mean_test_score"]
    n_features = np.arange(1, len(mean_scores) + 1)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.plot(
        n_features,
        mean_scores,
        marker="o",
        markersize=3.2,
        linewidth=1.2,
        color=COLORS["blue"],
        label=r"Mean CV $R^2$"
    )

    ax.axvline(
        optimal_k,
        color=COLORS["red"],
        linestyle="--",
        linewidth=1.1,
        label=f"Optimal: {optimal_k}"
    )

    ax.set_xlabel("Number of selected features")
    ax.set_ylabel(r"Cross-validation $R^2$")
    ax.set_title("RFECV feature selection")

    ax.legend(frameon=True, edgecolor="black", framealpha=1.0)
    format_axes(ax)

    fig.tight_layout(pad=0.4)
    save_figure(fig, "Fig4_rfecv_curve")
    plt.show()


def plot_feature_importance(imp_df, top_n=12):
    imp = imp_df.sort_values("rf_importance", ascending=True).tail(top_n).copy()
    
    # Заменяем названия признаков на LaTeX-обозначения (неразмеченные остаются без изменений)
    imp["feature_display"] = imp["feature"].map(lambda x: FEATURE_MAP.get(x, x))

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 1.35))

    bars = ax.barh(
        imp["feature_display"],  # Передаем столбцы с новым оформлением
        imp["rf_importance"],
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        height=0.68
    )

    xmax = imp["rf_importance"].max()
    ax.set_xlim(0, xmax * 1.28)

    for bar, value in zip(bars, imp["rf_importance"]):
        ax.text(
            value + xmax * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=7.5
        )

    ax.set_xlabel("Feature importance")
    ax.set_ylabel("Descriptor")
    ax.set_title("Random Forest feature importance")

    format_axes(ax)
    fig.tight_layout(pad=0.35)
    save_figure(fig, "Fig5_feature_importance_rf")
    plt.show()


def plot_model_comparison(results_df):
    res = results_df.sort_values("R2_transformed", ascending=True)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.85))

    bars = ax.barh(
        res["Model"],
        res["R2_transformed"],
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        height=0.68
    )

    xmax = max(res["R2_transformed"].max(), 0.1)
    ax.set_xlim(0, xmax * 1.25)

    for bar, value in zip(bars, res["R2_transformed"]):
        ax.text(
            value + xmax * 0.025,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=7.5
        )

    ax.set_xlabel(r"$R^2$")
    ax.set_ylabel("Model")
    ax.set_title("Model performance")

    format_axes(ax)
    fig.tight_layout(pad=0.35)
    save_figure(fig, "Fig6_model_comparison")
    plt.show()


def plot_parity(y_true, y_pred, name="rf"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)

    min_val = min(y_true.min(), y_pred.min())
    max_val = max(y_true.max(), y_pred.max())
    margin = 0.07 * (max_val - min_val)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH))

    ax.scatter(
        y_true,
        y_pred,
        s=22,
        color=COLORS["green"],
        edgecolor=COLORS["black"],
        linewidth=0.35,
        alpha=0.78
    )

    ax.plot(
        [min_val - margin, max_val + margin],
        [min_val - margin, max_val + margin],
        linestyle="--",
        linewidth=1.1,
        color=COLORS["red"],
        label="Ideal"
    )

    ax.text(
        0.06,
        0.94,
        rf"$R^2$ = {r2:.3f}" + "\n" +
        rf"MAE = {mae:.2f}",
        transform=ax.transAxes,
        va="top",
        ha="left",
        fontsize=8.5,
        bbox=dict(
            boxstyle="round,pad=0.25",
            facecolor="white",
            edgecolor="black",
            linewidth=0.6,
            alpha=0.92
        )
    )

    ax.set_xlabel(r"Experimental $\Delta S_M$" + "\n" + r"(J kg$^{-1}$ K$^{-1}$)")
    ax.set_ylabel(r"Predicted $\Delta S_M$" + "\n" + r"(J kg$^{-1}$ K$^{-1}$)")
    ax.set_title(r"Experimental vs predicted $\Delta S_M$")

    ax.set_xlim(min_val - margin, max_val + margin)
    ax.set_ylim(min_val - margin, max_val + margin)
    ax.set_aspect("equal", adjustable="box")

    ax.legend(frameon=True, edgecolor="black", framealpha=1.0, loc="lower right")
    format_axes(ax)

    fig.tight_layout(pad=0.35)
    save_figure(fig, f"Fig7_parity_plot_{name}")
    plt.show()


def plot_residuals(y_true, y_pred, name="rf"):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    residuals = y_pred - y_true

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.axhline(0, color=COLORS["black"], linewidth=0.9)

    ax.scatter(
        y_true,
        residuals,
        s=22,
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.35,
        alpha=0.78
    )

    ax.set_xlabel(r"Experimental $\Delta S_M$" + "\n" + r"(J kg$^{-1}$ K$^{-1}$)")
    ax.set_ylabel("Residual")
    ax.set_title("Residual analysis")

    format_axes(ax)
    fig.tight_layout(pad=0.35)
    save_figure(fig, f"Fig8_residuals_{name}")
    plt.show()


def plot_learning_curve(train_sizes_abs, train_mean, test_mean, train_std, test_std):
    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.plot(
        train_sizes_abs,
        train_mean,
        marker="o",
        markersize=3.2,
        linewidth=1.2,
        color=COLORS["blue"],
        label="Training"
    )

    ax.plot(
        train_sizes_abs,
        test_mean,
        marker="s",
        markersize=3.2,
        linewidth=1.2,
        color=COLORS["red"],
        label="CV"
    )

    ax.fill_between(
        train_sizes_abs,
        train_mean - train_std,
        train_mean + train_std,
        color=COLORS["blue"],
        alpha=0.15,
        linewidth=0
    )

    ax.fill_between(
        train_sizes_abs,
        test_mean - test_std,
        test_mean + test_std,
        color=COLORS["red"],
        alpha=0.15,
        linewidth=0
    )

    ax.set_xlabel("Training set size")
    ax.set_ylabel(r"$R^2$")
    ax.set_title("Learning curve")

    ax.set_ylim(0.0, 1.05)
    ax.legend(frameon=True, edgecolor="black", framealpha=1.0)
    format_axes(ax)

    fig.tight_layout(pad=0.35)
    save_figure(fig, "Fig9_learning_curve_rf")
    plt.show()


def compute_permutation_importance(model, X_test, y_test, feature_names,
                                    n_repeats=50, random_state=42):
    perm = permutation_importance(
        model, X_test, y_test,
        n_repeats=n_repeats,
        random_state=random_state,
        scoring="r2",
        n_jobs=-1
    )
    perm_df = pd.DataFrame({
        "feature": feature_names,
        "perm_importance_mean": perm.importances_mean,
        "perm_importance_std": perm.importances_std
    }).sort_values("perm_importance_mean", ascending=False)
    return perm_df


def plot_permutation_importance(perm_df, top_n=12):
    imp = perm_df.sort_values("perm_importance_mean", ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 1.35))

    bars = ax.barh(
        imp["feature"],
        imp["perm_importance_mean"],
        xerr=imp["perm_importance_std"],
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        height=0.68,
        capsize=2.5,
        error_kw={"linewidth": 0.8, "ecolor": COLORS["gray"]}
    )

    xmax = (imp["perm_importance_mean"] + imp["perm_importance_std"]).max()
    xmin = min(0, (imp["perm_importance_mean"] - imp["perm_importance_std"]).min())
    ax.set_xlim(xmin * 1.1 if xmin < 0 else 0, xmax * 1.3)

    for bar, value in zip(bars, imp["perm_importance_mean"]):
        ax.text(
            value + xmax * 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=7.5
        )

    ax.axvline(0, color=COLORS["black"], linewidth=0.7)
    ax.set_xlabel(r"Permutation importance ($\Delta R^2$, test set)")
    ax.set_ylabel("Descriptor")
    ax.set_title("Random Forest permutation importance")

    format_axes(ax)
    fig.tight_layout(pad=0.35)
    save_figure(fig, "Fig5_feature_importance_rf_permutation")
    plt.show()


def select_one_se_features(rfecv, min_features_to_select=1):
    cv_results = rfecv.cv_results_
    mean_scores = np.asarray(cv_results["mean_test_score"])
    std_scores = np.asarray(cv_results["std_test_score"])
    n_features = np.arange(min_features_to_select,
                            min_features_to_select + len(mean_scores))

    n_splits = rfecv.cv if isinstance(rfecv.cv, int) else 5
    se_scores = std_scores / np.sqrt(n_splits)

    best_idx = int(np.argmax(mean_scores))
    best_score = mean_scores[best_idx]
    best_se = se_scores[best_idx]
    threshold = best_score - best_se

    candidates = np.where(mean_scores >= threshold)[0]
    one_se_idx = int(candidates.min())

    return {
        "n_features": n_features,
        "mean_scores": mean_scores,
        "se_scores": se_scores,
        "best_idx": best_idx,
        "best_n": n_features[best_idx],
        "one_se_idx": one_se_idx,
        "one_se_n": n_features[one_se_idx],
        "threshold": threshold,
    }


def plot_rfecv_curve_one_se(rfecv, min_features_to_select=1):
    if rfecv is None:
        print("RFECV plot skipped: RFECV was not performed.")
        return None

    stats = select_one_se_features(rfecv, min_features_to_select)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 0.72))

    ax.plot(
        stats["n_features"],
        stats["mean_scores"],
        marker="o",
        markersize=2.6,
        linewidth=1.0,
        color=COLORS["blue"],
        label=r"Mean CV $R^2$"
    )

    ax.fill_between(
        stats["n_features"],
        stats["mean_scores"] - stats["se_scores"],
        stats["mean_scores"] + stats["se_scores"],
        color=COLORS["blue"],
        alpha=0.15,
        linewidth=0
    )

    ax.axhline(
        stats["threshold"],
        color=COLORS["gray"],
        linestyle=":",
        linewidth=1.0,
        label="Best score - 1 SE"
    )

    ax.axvline(
        stats["best_n"],
        color=COLORS["green"],
        linestyle="--",
        linewidth=1.0,
        label=f"Max score: {stats['best_n']}"
    )

    ax.axvline(
        stats["one_se_n"],
        color=COLORS["red"],
        linestyle="--",
        linewidth=1.2,
        label=f"1-SE optimal: {stats['one_se_n']}"
    )

    ax.set_xlabel("Number of selected features")
    ax.set_ylabel(r"Cross-validation $R^2$")
    ax.set_title("RFECV feature selection (1-SE rule)")

    ax.legend(frameon=True, edgecolor="black", framealpha=1.0, fontsize=7)
    format_axes(ax)

    fig.tight_layout(pad=0.4)
    save_figure(fig, "Fig4_rfecv_curve_one_se")
    plt.show()

    print(f"Best mean CV R^2 = {stats['mean_scores'][stats['best_idx']]:.4f} "
          f"at {stats['best_n']} features")
    print(f"1-SE rule selects {stats['one_se_n']} features "
          f"(threshold = {stats['threshold']:.4f})")

    return stats


def compute_all_models_importance(models, X_test, y_test, feature_names,
                                   n_repeats=50, random_state=42):
    importance_matrix = pd.DataFrame(index=feature_names)

    for name, model in models.items():
        perm = permutation_importance(
            model, X_test, y_test,
            n_repeats=n_repeats,
            random_state=random_state,
            scoring="r2",
            n_jobs=-1
        )
        importance_matrix[name] = perm.importances_mean

    return importance_matrix


def compute_all_models_native_importance(models, feature_names):
    native_matrix = pd.DataFrame(index=feature_names)

    for name, model in models.items():
        if hasattr(model, "feature_importances_"):
            native_matrix[name] = model.feature_importances_
        elif hasattr(model, "coef_"):
            native_matrix[name] = np.abs(np.ravel(model.coef_))
        elif hasattr(model, "estimators_"):
            sub_importances = [
                est.feature_importances_ for est in model.estimators_
                if hasattr(est, "feature_importances_")
            ]
            if sub_importances:
                native_matrix[name] = np.mean(sub_importances, axis=0)
            else:
                print(f"Skipping '{name}': no native importance available.")
        else:
            print(f"Skipping '{name}': no native importance available.")

    return native_matrix


def plot_single_model_importance(importance_series, model_name, method_label,
                                  save_dir, top_n=12, xlabel=None):
    imp = importance_series.sort_values(ascending=True).tail(top_n)

    fig, ax = plt.subplots(figsize=(COLUMN_WIDTH, COLUMN_WIDTH * 1.1))

    bars = ax.barh(
        imp.index,
        imp.values,
        color=COLORS["blue"],
        edgecolor=COLORS["black"],
        linewidth=0.45,
        height=0.68
    )

    xmax = imp.values.max()
    xmin = min(0, imp.values.min())
    ax.set_xlim(xmin * 1.1 if xmin < 0 else 0, xmax * 1.3)

    for bar, value in zip(bars, imp.values):
        ax.text(
            value + xmax * 0.03,
            bar.get_y() + bar.get_height() / 2,
            f"{value:.3f}",
            va="center",
            ha="left",
            fontsize=7.5
        )

    ax.axvline(0, color=COLORS["black"], linewidth=0.7)
    ax.set_xlabel(xlabel or method_label)
    ax.set_ylabel("Descriptor")
    ax.set_title(f"{model_name}: {method_label}")

    format_axes(ax)
    fig.tight_layout(pad=0.35)

    safe_model = model_name.lower().replace(" ", "_")
    safe_method = method_label.lower().replace(" ", "_")
    save_figure_to(fig, f"{safe_method}_{safe_model}", save_dir)
    plt.show()


def remove_outliers_2_5_sigma(y_true, y_pred, formulas=None, out_dir="."):
    # Принудительно выравниваем массивы в 1D векторы
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)
    
    # Остатки
    residuals = y_pred - y_true

    # Стандартное отклонение (с ddof=0, как по умолчанию в numpy.std)
    std = residuals.std()
    threshold = 2.5 * std
    
    is_outlier = np.abs(residuals) > threshold
    clean_mask = ~is_outlier  # Маска "чистых" данных

    r2_before = r2_score(y_true, y_pred)
    mae_before = mean_absolute_error(y_true, y_pred)
    r2_after = r2_score(y_true[clean_mask], y_pred[clean_mask])
    mae_after = mean_absolute_error(y_true[clean_mask], y_pred[clean_mask])

    print(f"\n--- Outlier removal (|error| > 2.5 sigma), Random Forest ---")
    print(f"Residual std = {std:.4f}, threshold = {threshold:.4f}")
    print(f"Removed {is_outlier.sum()} of {len(y_true)} points "
          f"({100 * is_outlier.sum() / len(y_true):.1f}%)")
    print(f"R2  before: {r2_before:.3f}  ->  after: {r2_after:.3f}")
    print(f"MAE before: {mae_before:.3f}  ->  after: {mae_after:.3f}")

    # Корректная фильтрация формул
    if formulas is not None:
        formulas_arr = np.ravel(formulas)
        outlier_formulas = formulas_arr[is_outlier]
    else:
        outlier_formulas = np.where(is_outlier)[0]

    outliers_df = pd.DataFrame({
        "formula": outlier_formulas,
        "y_true": y_true[is_outlier],
        "y_pred": y_pred[is_outlier],
        "residual": residuals[is_outlier]
    }).sort_values("residual", key=np.abs, ascending=False)

    # Сохраняем в Excel
    out_path = os.path.join(out_dir, "outliers_removed_2_5_sigma_random_forest.xlsx")
    outliers_df.to_excel(out_path, index=False)
    print(f"Excluded points saved to {out_path}")

    # Возвращаем словарь со ВСЕМИ возможными именами ключей для маски
    return {
        "clean_mask": clean_mask,
        "keep_mask": clean_mask,
        "is_outlier": is_outlier,
        "r2_before": r2_before,
        "r2_after": r2_after,
        "mae_before": mae_before,
        "mae_after": mae_after,
        "outliers_df": outliers_df
    }

# Обработка данных и признаков

In [6]:
os.makedirs(out_dir, exist_ok=True)
feature_importance_dir = os.path.join(out_dir, "feature_importance_by_model")
os.makedirs(feature_importance_dir, exist_ok=True)

# Загрузка Excel файла
df = pd.read_excel(file_path)
df.columns = df.columns.str.strip()

print(f"Initial dataset size: {df.shape}")
print("Columns:", list(df.columns))

target = "delta_S_M (J/kg K)"

if target not in df.columns:
    raise KeyError(f"Target column '{target}' not found.")

df[target] = pd.to_numeric(df[target], errors="coerce")
df = df.dropna(subset=[target]).copy()

if "H_ex (T)" in df.columns:
    df["H_ex (T)"] = pd.to_numeric(df["H_ex (T)"], errors="coerce")

print(f"After target cleaning: {len(df)} rows")

drop_cols = [
    "Chem formula",
    "Clean_formula",
    "Ref.",
    "T_c (K)",
    "delta_T_ad (K)",
    "RCP(S) (J/kg)",
    target
]

X = df.drop(columns=drop_cols, errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.median())

y_original = df[target].values
delta_s = pd.Series(y_original)
delta_s = delta_s[delta_s > 0]

print(f"Initial number of features: {X.shape[1]}")

if X.shape[1] == 0:
    raise ValueError("No numeric features found.")

# Фильтрация сильно коррелированных признаков (R > 0.95)
if X.shape[1] > 1:
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )

    to_drop = [
        column for column in upper.columns
        if any(upper[column] > 0.95)
    ]

    X = X.drop(columns=to_drop)

print(f"Number of features after correlation filtering: {X.shape[1]}")

Initial dataset size: (177, 508)
Columns: ['Chem formula', 'T_c (K)', 'delta_T_ad (K)', 'delta_S_M (J/kg K)', 'RCP(S) (J/kg)', 'H_ex (T)', 'Ref.', 'Clean_formula', 'H', 'He', 'Li', 'Be', 'B', 'C', 'N', 'O', 'F', 'Ne', 'Na', 'Mg', 'Al', 'Si', 'P', 'S', 'Cl', 'Ar', 'K', 'Ca', 'Sc', 'Ti', 'V', 'Cr', 'Mn', 'Fe', 'Co', 'Ni', 'Cu', 'Zn', 'Ga', 'Ge', 'As', 'Se', 'Br', 'Kr', 'Rb', 'Sr', 'Y', 'Zr', 'Nb', 'Mo', 'Tc', 'Ru', 'Rh', 'Pd', 'Ag', 'Cd', 'In', 'Sn', 'Sb', 'Te', 'I', 'Xe', 'Cs', 'Ba', 'La', 'Ce', 'Pr', 'Nd', 'Pm', 'Sm', 'Eu', 'Gd', 'Tb', 'Dy', 'Ho', 'Er', 'Tm', 'Yb', 'Lu', 'Hf', 'Ta', 'W', 'Re', 'Os', 'Ir', 'Pt', 'Au', 'Hg', 'Tl', 'Pb', 'Bi', 'Po', 'At', 'Rn', 'Fr', 'Ra', 'Ac', 'Th', 'Pa', 'U', 'Np', 'Pu', 'ave:atomic_number', 'ave:atomic_radius', 'ave:atomic_radius_rahm', 'ave:atomic_volume', 'ave:atomic_weight', 'ave:boiling_point', 'ave:bulk_modulus', 'ave:c6_gb', 'ave:covalent_radius_cordero', 'ave:covalent_radius_pyykko', 'ave:covalent_radius_pyykko_double', 'ave:covalent_radius_pyy

# Преобразование целевой переменной, масштабирование и деление выборки

In [7]:
skewness = pd.Series(y_original).skew()

if abs(skewness) > 1:
    print(f"Target distribution is skewed (skew = {skewness:.2f}). Applying log1p transformation.")
    y = np.log1p(y_original)
    target_transformed = True
else:
    print(f"Target distribution is not strongly skewed (skew = {skewness:.2f}).")
    y = y_original.copy()
    target_transformed = False

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

Target distribution is skewed (skew = 2.38). Applying log1p transformation.


# Отбор признаков через RFECV и обучение моделей

In [8]:
# ============================================================
# МАТРИЦА: для каждой из 6 моделей — свой RFECV (Шаг А),
# затем GridSearchCV гиперпараметров на найденном наборе
# признаков (Шаг Б). Данные — исходные, с выбросами (X_scaled, y).
# Не вложенная CV (один и тот же cv для отбора признаков и
# подбора гиперпараметров) — сделано осознанно, см. пояснение выше.
# ============================================================

model_matrix_dir = os.path.join(out_dir, "model_matrix")
os.makedirs(model_matrix_dir, exist_ok=True)

# --- Шаг А: базовые ("дефолтные") версии моделей для RFECV ---
rfecv_base_estimators = {
    "Ridge": Ridge(alpha=1.0),
    "Lasso": Lasso(alpha=0.01, max_iter=10000),
    "ElasticNet": ElasticNet(alpha=0.01, l1_ratio=0.5, max_iter=10000),
    "Random Forest": RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1),
    "XGBoost": xgb.XGBRegressor(
        n_estimators=100, max_depth=4, learning_rate=0.05,
        random_state=42, verbosity=0, objective="reg:squarederror"
    ),
    "CatBoost": CatBoostRegressor(
        iterations=300, depth=6, learning_rate=0.05,
        random_state=42, verbose=False
    ),
}

# --- Шаг Б: сетки гиперпараметров для GridSearchCV ---
param_grids = {
    "Ridge": {"alpha": np.logspace(-2, 2, 10)},
    "Lasso": {"alpha": np.logspace(-3, 1, 10)},
    "ElasticNet": {
        "alpha": np.logspace(-3, 1, 6),
        "l1_ratio": [0.1, 0.3, 0.5, 0.7, 0.9, 1.0]
    },
    "Random Forest": {
        "n_estimators": [100, 200, 300],
        "max_depth": [5, 7, 10, None],
        "min_samples_leaf": [2, 4, 6]
    },
    "XGBoost": {
        "n_estimators": [100, 200, 300],
        "max_depth": [3, 5, 7],
        "learning_rate": [0.03, 0.05, 0.1]
    },
    "CatBoost": {
        "iterations": [200, 300, 400],
        "depth": [4, 6, 8],
        "learning_rate": [0.03, 0.05, 0.1]
    },
}

# --- конструкторы "чистых" моделей для GridSearchCV на Шаге Б ---
def make_fresh_model(name):
    if name == "Ridge":
        return Ridge()
    if name == "Lasso":
        return Lasso(max_iter=10000)
    if name == "ElasticNet":
        return ElasticNet(max_iter=10000)
    if name == "Random Forest":
        return RandomForestRegressor(random_state=42, n_jobs=-1)
    if name == "XGBoost":
        return xgb.XGBRegressor(random_state=42, n_jobs=-1, verbosity=0, objective="reg:squarederror")
    if name == "CatBoost":
        return CatBoostRegressor(random_state=42, verbose=False)
    raise ValueError(name)


model_matrix_results = []
model_matrix_features = {}

model_matrix_estimators = {}
model_matrix_masks = {}

for model_name, base_estimator in rfecv_base_estimators.items():
    print(f"\n=== {model_name}: Шаг А — RFECV ===")

    model_rfecv = RFECV(
        estimator=base_estimator,
        step=1,
        cv=cv,
        scoring="r2",
        min_features_to_select=1,
        n_jobs=-1 if model_name not in ("CatBoost",) else None,  # CatBoost плохо дружит с n_jobs здесь
    )
    model_rfecv.fit(X_scaled, y)

    n_features_A = model_rfecv.n_features_
    mask_A = model_rfecv.support_
    features_A = list(X.columns[mask_A])
    X_scaled_A = X_scaled[:, mask_A]

    model_matrix_features[model_name] = features_A

    print(f"{model_name}: RFECV выбрал {n_features_A} признаков")

    print(f"=== {model_name}: Шаг Б — GridSearchCV на {n_features_A} признаках ===")

    grid = GridSearchCV(
        estimator=make_fresh_model(model_name),
        param_grid=param_grids[model_name],
        cv=cv,
        scoring="r2",
        n_jobs=-1,
    )
    grid.fit(X_scaled_A, y)

    model_matrix_estimators[model_name] = grid.best_estimator_
    model_matrix_masks[model_name] = mask_A

    best_estimator_B = grid.best_estimator_
    best_params_B = grid.best_params_
    r2_transformed = grid.best_score_

    # предсказания той же (не вложенной) CV для MAE в трансформированной шкале
    # и для метрик в физических единицах
    y_pred_cv = cross_val_predict(best_estimator_B, X_scaled_A, y, cv=cv)
    mae_transformed = mean_absolute_error(y, y_pred_cv)

    if target_transformed:
        y_orig_matrix = np.expm1(y)
        y_pred_cv_orig = np.expm1(y_pred_cv)
    else:
        y_orig_matrix = y.copy() if isinstance(y, np.ndarray) else y.values.copy()
        y_pred_cv_orig = y_pred_cv.copy()

    r2_physical = r2_score(y_orig_matrix, y_pred_cv_orig)
    mae_physical = mean_absolute_error(y_orig_matrix, y_pred_cv_orig)

    print(f"{model_name}: best params = {best_params_B}")
    print(f"{model_name}: R2 (transformed) = {r2_transformed:.4f}, "
          f"R2 (physical) = {r2_physical:.4f}")

    model_matrix_results.append({
        "Model": model_name,
        "N_features": n_features_A,
        "Best_hyperparameters": str(best_params_B),
        "R2_CV_transformed": r2_transformed,
        "MAE_CV_transformed": mae_transformed,
        "R2_CV_physical": r2_physical,
        "MAE_CV_physical": mae_physical,
    })

model_matrix_df = pd.DataFrame(model_matrix_results).sort_values(
    "R2_CV_physical", ascending=False
)

print("\n\n=== ИТОГОВАЯ МАТРИЦА: модель | число признаков | гиперпараметры | метрики ===")
print(model_matrix_df.to_string(index=False))

# --- сохранение результатов ---
model_matrix_df.to_excel(
    os.path.join(model_matrix_dir, "model_matrix_summary.xlsx"), index=False
)

with pd.ExcelWriter(os.path.join(model_matrix_dir, "model_matrix_features.xlsx")) as writer:
    for model_name, feats in model_matrix_features.items():
        pd.DataFrame({"feature": feats}).to_excel(
            writer, sheet_name=model_name[:31], index=False
        )

print(f"\nМатрица и списки признаков сохранены в {model_matrix_dir}")


=== Ridge: Шаг А — RFECV ===
Ridge: RFECV выбрал 1 признаков
=== Ridge: Шаг Б — GridSearchCV на 1 признаках ===
Ridge: best params = {'alpha': np.float64(4.6415888336127775)}
Ridge: R2 (transformed) = 0.3462, R2 (physical) = 0.2794

=== Lasso: Шаг А — RFECV ===
Lasso: RFECV выбрал 61 признаков
=== Lasso: Шаг Б — GridSearchCV на 61 признаках ===
Lasso: best params = {'alpha': np.float64(0.007742636826811269)}
Lasso: R2 (transformed) = 0.4708, R2 (physical) = 0.4374

=== ElasticNet: Шаг А — RFECV ===
ElasticNet: RFECV выбрал 15 признаков
=== ElasticNet: Шаг Б — GridSearchCV на 15 признаках ===
ElasticNet: best params = {'alpha': np.float64(0.00630957344480193), 'l1_ratio': 0.1}
ElasticNet: R2 (transformed) = 0.5286, R2 (physical) = 0.4381

=== Random Forest: Шаг А — RFECV ===
Random Forest: RFECV выбрал 93 признаков
=== Random Forest: Шаг Б — GridSearchCV на 93 признаках ===
Random Forest: best params = {'max_depth': 7, 'min_samples_leaf': 2, 'n_estimators': 300}
Random Forest: R2 (tran

# Оценка метрик качества и удаление выбросов

In [10]:
# ============================================================
# Для КАЖДОЙ из 6 моделей: своя диагностика выбросов (2.5σ на
# честных CV-остатках), свой очищенный датасет, переобучение
# GridSearchCV на прежних (по RFECV) признаках, но уже на
# чистых данных. df/X/y/X_scaled глобально НЕ меняются.
# ============================================================

from sklearn.base import clone

per_model_clean_dir = os.path.join(out_dir, "model_matrix_per_model_clean")
os.makedirs(per_model_clean_dir, exist_ok=True)

drop_cols = [
    "Chem formula", "Clean_formula", "Ref.", "T_c (K)",
    "delta_T_ad (K)", "RCP(S) (J/kg)", target
]

per_model_clean_results = []
per_model_outliers = {}

for model_name in model_matrix_features.keys():
    print(f"\n\n========== {model_name} ==========")

    feature_list = model_matrix_features[model_name]
    mask = X.columns.isin(feature_list)
    X_model = X_scaled[:, mask]

    fresh_model = clone(model_matrix_estimators[model_name])
    y_pred_cv = cross_val_predict(fresh_model, X_model, y, cv=cv)

    if target_transformed:
        y_orig_m = np.expm1(y)
        y_pred_cv_orig = np.expm1(y_pred_cv)
    else:
        y_orig_m = y.copy() if isinstance(y, np.ndarray) else y.values.copy()
        y_pred_cv_orig = y_pred_cv.copy()

    residuals = y_pred_cv_orig - y_orig_m
    std = residuals.std()
    threshold = 2.5 * std
    is_outlier = np.abs(residuals) > threshold

    print(f"{model_name}: residual std = {std:.4f}, threshold = {threshold:.4f}")
    print(f"{model_name}: {is_outlier.sum()} of {len(y)} points flagged as outliers")

    outlier_labels = df.index.values[is_outlier]
    outlier_formulas = (df["Clean_formula"].values[is_outlier]
                         if "Clean_formula" in df.columns else outlier_labels)
    per_model_outliers[model_name] = list(outlier_formulas)

    if is_outlier.sum() > 0:
        print("Excluded points:")
        print(pd.DataFrame({
            "formula": outlier_formulas,
            "y_true": y_orig_m[is_outlier],
            "residual": residuals[is_outlier]
        }).to_string(index=False))

    # --- новый датасет без выбросов именно этой модели ---
    df_model_clean = df.drop(index=outlier_labels).copy()
    print(f"Dataset size after outlier removal for {model_name}: "
          f"{len(df_model_clean)} rows (was {len(df)})")

    # --- пересборка X/y/X_scaled с нуля из df_model_clean ---
    X_model_clean = df_model_clean.drop(columns=drop_cols, errors="ignore")
    X_model_clean = X_model_clean.select_dtypes(include=[np.number])
    X_model_clean = X_model_clean.fillna(X_model_clean.median())

    y_original_model_clean = df_model_clean[target].values

    if X_model_clean.shape[1] > 1:
        corr_matrix = X_model_clean.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        to_drop = [c for c in upper.columns if any(upper[c] > 0.95)]
        X_model_clean = X_model_clean.drop(columns=to_drop)

    skew_model_clean = pd.Series(y_original_model_clean).skew()
    if abs(skew_model_clean) > 1:
        y_model_clean = np.log1p(y_original_model_clean)
        target_transformed_model_clean = True
    else:
        y_model_clean = y_original_model_clean.copy()
        target_transformed_model_clean = False

    scaler_model_clean = StandardScaler()
    X_scaled_model_clean = scaler_model_clean.fit_transform(X_model_clean)
    cv_model_clean = KFold(n_splits=5, shuffle=True, random_state=42)

    # --- те же признаки (по именам), но уже на чистых данных ---
    feature_list_present = [f for f in feature_list if f in X_model_clean.columns]
    missing = set(feature_list) - set(feature_list_present)
    if missing:
        print(f"ВНИМАНИЕ: признаки пропали после чистки: {missing}")

    mask_clean = X_model_clean.columns.isin(feature_list_present)
    X_final = X_scaled_model_clean[:, mask_clean]

    # --- GridSearchCV заново, на чистых данных ---
    grid_clean = GridSearchCV(
        estimator=make_fresh_model(model_name),
        param_grid=param_grids[model_name],
        cv=cv_model_clean,
        scoring="r2",
        n_jobs=-1,
    )
    grid_clean.fit(X_final, y_model_clean)

    best_clean = grid_clean.best_estimator_
    best_params_clean = grid_clean.best_params_
    r2_transformed_clean = grid_clean.best_score_

    y_pred_cv_clean = cross_val_predict(best_clean, X_final, y_model_clean, cv=cv_model_clean)
    mae_transformed_clean = mean_absolute_error(y_model_clean, y_pred_cv_clean)

    if target_transformed_model_clean:
        y_orig_final = np.expm1(y_model_clean)
        y_pred_final_orig = np.expm1(y_pred_cv_clean)
    else:
        y_orig_final = y_model_clean.copy()
        y_pred_final_orig = y_pred_cv_clean.copy()

    r2_physical_clean = r2_score(y_orig_final, y_pred_final_orig)
    mae_physical_clean = mean_absolute_error(y_orig_final, y_pred_final_orig)

    print(f"{model_name} (clean): N features used = {mask_clean.sum()}, "
          f"best params = {best_params_clean}")
    print(f"{model_name} (clean): R2 (transformed) = {r2_transformed_clean:.4f}, "
          f"R2 (physical) = {r2_physical_clean:.4f}")

    per_model_clean_results.append({
        "Model": model_name,
        "N_outliers_removed": int(is_outlier.sum()),
        "Dataset_size_after_cleaning": len(df_model_clean),
        "N_features": int(mask_clean.sum()),
        "Best_hyperparameters": str(best_params_clean),
        "R2_CV_transformed": r2_transformed_clean,
        "MAE_CV_transformed": mae_transformed_clean,
        "R2_CV_physical": r2_physical_clean,
        "MAE_CV_physical": mae_physical_clean,
    })

per_model_clean_df = pd.DataFrame(per_model_clean_results).sort_values(
    "R2_CV_physical", ascending=False
)

print("\n\n=== ИТОГОВОЕ СРАВНЕНИЕ (каждая модель — на СВОЁМ очищенном датасете) ===")
print(per_model_clean_df.to_string(index=False))

per_model_clean_df.to_excel(
    os.path.join(per_model_clean_dir, "per_model_clean_summary.xlsx"), index=False
)

print(f"\nСохранено в {per_model_clean_dir}")



========== Ridge ==========
Ridge: residual std = 2.3643, threshold = 5.9107
Ridge: 4 of 175 points flagged as outliers
Excluded points:
         formula  y_true   residual
  La0.7Ca0.3MnO3     9.4  -7.652606
La0.67Sr0.33MnO3    14.8 -12.887730
 La1.4Ca1.6Mn2O7    16.8 -12.275881
       Gd5Si2Ge2    18.4 -13.875881
Dataset size after outlier removal for Ridge: 171 rows (was 175)
Ridge (clean): N features used = 1, best params = {'alpha': np.float64(1.6681005372000592)}
Ridge (clean): R2 (transformed) = 0.3776, R2 (physical) = 0.4369


========== Lasso ==========
Lasso: residual std = 2.1008, threshold = 5.2520
Lasso: 5 of 175 points flagged as outliers
Excluded points:
                   formula  y_true   residual
     La0.7Ca0.25Sr0.05MnO3   10.50  -5.668187
La0.67Ba0.33Mn0.98Ti0.02O3    6.34  -5.331767
            La0.7Ca0.3MnO3    9.40  -7.235866
          La0.67Sr0.33MnO3   14.80 -13.184493
           La1.4Ca1.6Mn2O7   16.80 -12.123599
Dataset size after outlier removal for Lasso

# Переобучение моделей и расчет метрик

In [ ]:
outlier_df_labels = outlier_flags.loc[outlier_flags["n_models_flagging"] >= 1, "df_label"].tolist()

print(f"Выбросы хотя бы для ОДНОЙ модели: {len(outlier_df_labels)} точек")
print(outlier_flags.loc[
    outlier_flags["df_label"].isin(outlier_df_labels),
    ["formula", "y_true", "n_models_flagging"]
].to_string(index=False))

In [ ]:
# ============================================================
# Удаление выбросов из df + ПОЛНАЯ пересборка X/y/X_scaled/cv
# прямо здесь же (дословно повторяет логику ячеек "Обработка
# данных и признаков" и "Преобразование...") — чтобы не
# зависеть от повторного запуска ячеек, стоящих выше.
# ============================================================

df = df.drop(index=outlier_df_labels)
print(f"Dataset size after outlier removal: {len(df)} rows")

drop_cols = [
    "Chem formula",
    "Clean_formula",
    "Ref.",
    "T_c (K)",
    "delta_T_ad (K)",
    "RCP(S) (J/kg)",
    target
]

X = df.drop(columns=drop_cols, errors="ignore")
X = X.select_dtypes(include=[np.number])
X = X.fillna(X.median())

y_original = df[target].values
delta_s = pd.Series(y_original)
delta_s = delta_s[delta_s > 0]

print(f"Number of features before correlation filtering: {X.shape[1]}")

if X.shape[1] > 1:
    corr_matrix = X.corr().abs()
    upper = corr_matrix.where(
        np.triu(np.ones(corr_matrix.shape), k=1).astype(bool)
    )
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    X = X.drop(columns=to_drop)

print(f"Number of features after correlation filtering: {X.shape[1]}")

skewness = pd.Series(y_original).skew()

if abs(skewness) > 1:
    print(f"Target distribution is skewed (skew = {skewness:.2f}). Applying log1p transformation.")
    y = np.log1p(y_original)
    target_transformed = True
else:
    print(f"Target distribution is not strongly skewed (skew = {skewness:.2f}).")
    y = y_original.copy()
    target_transformed = False

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

cv = KFold(n_splits=5, shuffle=True, random_state=42)

print(f"\nFinal shapes after cleaning: X={X.shape}, X_scaled={X_scaled.shape}, y={y.shape}")

In [ ]:
# ============================================================
# Переобучение на уже найденных ранее признаках (без RFECV) —
# только Шаг Б (GridSearchCV) на признаках, отобранных ДО
# удаления выбросов. Данные — очищенные (X_scaled, y уже
# пересобраны из очищенного df).
# ============================================================

model_matrix_clean_dir = os.path.join(out_dir, "model_matrix_clean")
os.makedirs(model_matrix_clean_dir, exist_ok=True)

model_matrix_clean_results = []
model_matrix_clean_estimators = {}
model_matrix_clean_masks = {}

for model_name, feature_list in model_matrix_features.items():
    print(f"\n=== {model_name}: GridSearchCV на ранее отобранных признаках ===")

    mask = X.columns.isin(feature_list)

    if mask.sum() != len(feature_list):
        missing = set(feature_list) - set(X.columns[mask])
        print(f"ВНИМАНИЕ: {len(missing)} признак(ов) из старого набора "
              f"отсутствуют в новом X: {missing}")

    X_model = X_scaled[:, mask]
    n_features_used = mask.sum()

    grid = GridSearchCV(
        estimator=make_fresh_model(model_name),
        param_grid=param_grids[model_name],
        cv=cv,
        scoring="r2",
        n_jobs=-1,
    )
    grid.fit(X_model, y)

    best_estimator = grid.best_estimator_
    best_params = grid.best_params_
    r2_transformed = grid.best_score_

    y_pred_cv = cross_val_predict(best_estimator, X_model, y, cv=cv)
    mae_transformed = mean_absolute_error(y, y_pred_cv)

    if target_transformed:
        y_orig_clean_matrix = np.expm1(y)
        y_pred_cv_orig = np.expm1(y_pred_cv)
    else:
        y_orig_clean_matrix = y.copy() if isinstance(y, np.ndarray) else y.values.copy()
        y_pred_cv_orig = y_pred_cv.copy()

    r2_physical = r2_score(y_orig_clean_matrix, y_pred_cv_orig)
    mae_physical = mean_absolute_error(y_orig_clean_matrix, y_pred_cv_orig)

    print(f"{model_name}: {n_features_used} признаков, best params = {best_params}")
    print(f"{model_name}: R2 (transformed) = {r2_transformed:.4f}, "
          f"R2 (physical) = {r2_physical:.4f}")

    model_matrix_clean_estimators[model_name] = best_estimator
    model_matrix_clean_masks[model_name] = mask

    model_matrix_clean_results.append({
        "Model": model_name,
        "N_features": int(n_features_used),
        "Best_hyperparameters": str(best_params),
        "R2_CV_transformed": r2_transformed,
        "MAE_CV_transformed": mae_transformed,
        "R2_CV_physical": r2_physical,
        "MAE_CV_physical": mae_physical,
    })

model_matrix_clean_df = pd.DataFrame(model_matrix_clean_results).sort_values(
    "R2_CV_physical", ascending=False
)

print("\n\n=== ИТОГОВАЯ МАТРИЦА (очищенные данные, признаки не искались заново) ===")
print(model_matrix_clean_df.to_string(index=False))

model_matrix_clean_df.to_excel(
    os.path.join(model_matrix_clean_dir, "model_matrix_clean_summary.xlsx"), index=False
)

print(f"\nСохранено в {model_matrix_clean_dir}")

# Расчет кривой обучения (Learning Curve)

In [ ]:
train_sizes_abs, train_scores, test_scores = learning_curve(
    best_rf_clean,       # Модель, обученная на чистых данных
    X_clean,             # Признаки очищенного датасета
    y_clean,             # Целевая переменная очищенного датасета
    cv=cv_clean,         # Объект KFold для кросс-валидации
    scoring="r2",
    train_sizes=np.linspace(0.2, 1.0, 6),
    n_jobs=-1,
    random_state=42
)

train_mean = train_scores.mean(axis=1)
train_std = train_scores.std(axis=1)
test_mean = test_scores.mean(axis=1)
test_std = test_scores.std(axis=1)

# Отрисовка всех основных графиков

In [ ]:
plot_delta_s_histogram(delta_s)
plot_transformation_histogram(y_orig)

rf_importances = best_rf_clean.feature_importances_

# 2. Собираем аккуратный DataFrame
rf_imp_df = pd.DataFrame({
    "feature": selected_features,
    "rf_importance": rf_importances
}).sort_values("rf_importance", ascending=False)

# 3. Строим график
plot_feature_importance(rf_imp_df, top_n=12)

# RFECV
_min_feat = 1
one_se_stats = plot_rfecv_curve_one_se(rfecv, min_features_to_select=_min_feat)

# Сравнение моделей на чистых данных
plot_model_comparison(results_clean_df)

# Parity & Residual plots (на очищенной выборке)
plot_parity(y_clean_orig, predictions_clean_orig["Random Forest"], name="random_forest")
plot_residuals(y_clean_orig, predictions_clean_orig["Random Forest"], name="random_forest")

plot_parity(y_clean_orig, predictions_clean_orig["XGBoost"], name="xgboost")
plot_residuals(y_clean_orig, predictions_clean_orig["XGBoost"], name="xgboost")

plot_parity(y_clean_orig, predictions_clean_orig["Voting"], name="voting")

# Кривая обучения
plot_learning_curve(
    train_sizes_abs,
    train_mean,
    test_mean,
    train_std,
    test_std
)

# Сохранение финальных таблиц Excel

In [ ]:
# Сохранение финальных таблиц Excel
results_df.to_excel(os.path.join(out_dir, "model_metrics_transformed.xlsx"), index=False)
results_orig_df.to_excel(os.path.join(out_dir, "model_metrics_original_units.xlsx"), index=False)

# Сохраняем актуальную важность признаков из Random Forest
rf_imp_df.to_excel(os.path.join(out_dir, "feature_importance_random_forest.xlsx"), index=False)

print(f"\nAll figures and tables saved to folder: {out_dir}")